# 02: Benchmark Retrieval, Reranker, Graph

Цель: прогнать единый benchmark для всех retrieval-методов на одном `eval_queries.jsonl`/`qrels.tsv`, включая:
- BM25
- dense MiniLM
- MedCPT zero-shot
- trained MedCPT bi-encoder
- hybrid variants
- RAG Fusion variants
- graph-aware expansion поверх run-файлов
- optional Cross-Encoder reranking для выбранных baseline runs

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

import pandas as pd

REPO_OWNER = "terrylimax"
REPO_NAME = "medical-rag-reranker"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
BRANCH = "main"

RUN_ID = os.environ.get("COLAB_RUN_ID", "medquad_full_v1")
USE_DRIVE = True
TOP_K = 50
KS = "1,5,10,20,50"

PROJECT_DIR = Path("/content") / REPO_NAME
PROJECT_PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

In [ ]:
def mount_drive_or_content(use_drive: bool) -> Path:
    if not use_drive:
        return Path("/content") / "medical-rag-reranker-colab"

    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        return Path("/content/drive/MyDrive") / "medical-rag-reranker-colab"
    except Exception as exc:
        print("Drive mount failed; falling back to /content.")
        print(type(exc).__name__, str(exc))
        return Path("/content") / "medical-rag-reranker-colab"


DRIVE_BASE = mount_drive_or_content(USE_DRIVE)
DRIVE_ROOT = DRIVE_BASE / RUN_ID
MANIFEST_PATH = DRIVE_ROOT / "manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Run notebook 01 first; missing manifest: {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
INDEX_ROOT = ARTIFACT_ROOT / "indices"
RUN_ROOT = DRIVE_ROOT / "runs"
RETRIEVAL_RUN_DIR = RUN_ROOT / "retrieval"
RERANK_RUN_DIR = RUN_ROOT / "reranked"
SUMMARY_DIR = RUN_ROOT / "summaries"
PROCESSED_DRIVE_DIR = Path(manifest["data"]["processed_drive_dir"])

for path in [RETRIEVAL_RUN_DIR, RERANK_RUN_DIR, SUMMARY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Manifest loaded:", MANIFEST_PATH)

In [ ]:
def sh(args, *, cwd: Path | None = None) -> None:
    args = [str(arg) for arg in args]
    print("+", " ".join(args))
    subprocess.run(args, cwd=None if cwd is None else str(cwd), check=True)


def run_project(command: str, overrides: list[str]) -> None:
    payload = json.dumps(overrides, ensure_ascii=False)
    sh(
        [
            sys.executable,
            "-m",
            "medical_rag_reranker.commands",
            command,
            "--overrides",
            payload,
        ],
        cwd=PROJECT_DIR,
    )


def base_overrides() -> list[str]:
    return [
        "data.use_dvc=false",
        "data.processed_dir=data/processed",
        f"paths.artifacts_dir={INDEX_ROOT}",
        f"paths.runs_dir={RUN_ROOT}",
        "run.prep_data.out_dir=data/processed",
    ]

In [ ]:
if not (PROJECT_DIR / ".git").exists():
    sh(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR])
else:
    sh(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    sh(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    sh(["git", "pull", "--ff-only"], cwd=PROJECT_DIR)

sh(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    cwd=PROJECT_DIR,
)
packages = [
    "transformers",
    "datasets",
    "tokenizers",
    "hydra-core",
    "omegaconf",
    "fire",
    "rank-bm25",
    "mlflow",
    "sentence-transformers",
    "pandas",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "tqdm",
    "sentencepiece",
    "accelerate",
    "lightning",
    "torchmetrics",
]
sh([sys.executable, "-m", "pip", "install", "-q", *packages], cwd=PROJECT_DIR)

if not (PROJECT_DIR / "medical_rag_reranker").exists():
    raise FileNotFoundError(
        "Project package directory was not found. "
        f"Expected: {PROJECT_DIR / 'medical_rag_reranker'}"
    )
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project package import path:", PROJECT_DIR)

if PROJECT_PROCESSED_DIR.exists():
    shutil.rmtree(PROJECT_PROCESSED_DIR)
shutil.copytree(PROCESSED_DRIVE_DIR, PROJECT_PROCESSED_DIR)
print("Restored processed data:", PROJECT_PROCESSED_DIR)

## Retrieval benchmark

Все методы пишут TREC run-файл и `*.metrics.json`. Для RAG Fusion используется тот же индекс, что и у базового retriever, но retrieval layer расширяет запросы и агрегирует ранги через RRF.

In [ ]:
indices = manifest["indices"]


def metric_path_for_run(run_file: Path) -> Path:
    return run_file.with_suffix(run_file.suffix + ".metrics.json")


def load_metrics(run_file: Path) -> dict:
    path = metric_path_for_run(run_file)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))


def eval_retrieval_method(
    *,
    method: str,
    retrieval_config: str,
    index_path: Path,
    extra: list[str] | None = None,
) -> dict:
    run_file = RETRIEVAL_RUN_DIR / f"{method}.trec"
    overrides = base_overrides()
    overrides += [
        f"retrieval={retrieval_config}",
        f"retrieval_run.index={index_path}",
        f"retrieval_run.queries={PROJECT_PROCESSED_DIR / 'eval_queries.jsonl'}",
        f"retrieval_run.out={run_file}",
        f"retrieval_run.top_k={TOP_K}",
        f"retrieval_run.run_name={method}",
        f"run.eval_retrieval.eval_queries={PROJECT_PROCESSED_DIR / 'eval_queries.jsonl'}",
        f"run.eval_retrieval.qrels={PROJECT_PROCESSED_DIR / 'qrels.tsv'}",
        f"run.eval_retrieval.out_run={run_file}",
        f"run.eval_retrieval.run_path={run_file}",
        f'run.eval_retrieval.ks="{KS}"',
        f"run.eval_retrieval.run_name={method}",
        f"run.eval_retrieval.retriever={method}",
    ]
    if extra:
        overrides += extra

    run_project("retrieval_run", overrides)
    run_project("eval_retrieval", overrides)

    metrics = load_metrics(run_file)
    metrics.update(
        {
            "method": method,
            "retrieval_config": retrieval_config,
            "index_path": str(index_path),
            "run_file": str(run_file),
            "stage": "retrieval",
        }
    )
    return metrics

In [ ]:
method_specs = [
    {
        "method": "bm25",
        "retrieval_config": "bm25",
        "index_path": Path(indices["bm25"]["path"]),
    },
    {
        "method": "dense_minilm",
        "retrieval_config": "dense",
        "index_path": Path(indices["dense_minilm"]["path"]),
        "extra": ["retrieval.name=dense"],
    },
    {
        "method": "medcpt_zero_shot",
        "retrieval_config": "medcpt",
        "index_path": Path(indices["medcpt_zero_shot"]["path"]),
    },
    {
        "method": "hybrid_minilm",
        "retrieval_config": "hybrid",
        "index_path": Path(indices["hybrid_minilm"]["path"]),
    },
    {
        "method": "hybrid_medcpt_zero_shot",
        "retrieval_config": "hybrid_medcpt",
        "index_path": Path(indices["hybrid_medcpt_zero_shot"]["path"]),
    },
    {
        "method": "rag_fusion_bm25",
        "retrieval_config": "rag_fusion_bm25",
        "index_path": Path(indices["bm25"]["path"]),
    },
    {
        "method": "rag_fusion_dense_minilm",
        "retrieval_config": "rag_fusion_dense",
        "index_path": Path(indices["dense_minilm"]["path"]),
        "extra": ["retrieval.name=rag_fusion_dense_minilm"],
    },
    {
        "method": "rag_fusion_medcpt_zero_shot",
        "retrieval_config": "rag_fusion_medcpt_pilot",
        "index_path": Path(indices["medcpt_zero_shot"]["path"]),
        "extra": [
            "retrieval.name=rag_fusion_medcpt_zero_shot",
            "retrieval.base_retriever=bi_encoder",
        ],
    },
]

if "trained_medcpt" in indices:
    method_specs.extend(
        [
            {
                "method": "trained_medcpt",
                "retrieval_config": "medcpt",
                "index_path": Path(indices["trained_medcpt"]["path"]),
            },
            {
                "method": "hybrid_trained_medcpt",
                "retrieval_config": "hybrid_medcpt",
                "index_path": Path(indices["hybrid_trained_medcpt"]["path"]),
            },
            {
                "method": "rag_fusion_trained_medcpt",
                "retrieval_config": "rag_fusion_medcpt_pilot",
                "index_path": Path(indices["trained_medcpt"]["path"]),
                "extra": [
                    "retrieval.name=rag_fusion_trained_medcpt",
                    "retrieval.base_retriever=bi_encoder",
                ],
            },
        ]
    )

retrieval_rows = []
for spec in method_specs:
    print("\n== Benchmark", spec["method"], "==")
    retrieval_rows.append(eval_retrieval_method(**spec))

retrieval_df = pd.DataFrame(retrieval_rows)
display_cols = [
    "method",
    "NDCG@10",
    "R@10",
    "P@10",
    "Hit@10",
    "MRR@10",
    "latency_mean_ms",
    "latency_p95_ms",
    "index_size_mb",
    "run_file",
]
display(
    retrieval_df[
        [col for col in display_cols if col in retrieval_df.columns]
    ].sort_values("NDCG@10", ascending=False)
)

## Graph-aware expansion

Метод берёт готовый TREC run от base retriever, расширяет кандидатов соседними документами из metadata graph и агрегирует score через RRF. Это не заменяет BM25/dense, а проверяет, помогает ли графовая связность по теме/intent/group вытянуть релевантный документ выше.

In [ ]:
from collections import defaultdict


def read_trec_run(path: Path) -> dict[str, list[tuple[str, float]]]:
    runs: dict[str, list[tuple[str, float]]] = defaultdict(list)
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            qid, _q0, doc_id, _rank, score, _run_name = line.split()[:6]
            runs[qid].append((doc_id, float(score)))
    for qid in list(runs):
        runs[qid].sort(key=lambda item: item[1], reverse=True)
    return dict(runs)


def write_trec_run(
    path: Path, runs: dict[str, list[tuple[str, float]]], run_name: str
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for qid, docs in runs.items():
            for rank, (doc_id, score) in enumerate(docs, start=1):
                handle.write(f"{qid}\tQ0\t{doc_id}\t{rank}\t{score:.8f}\t{run_name}\n")


def load_graph_index(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("format") != "medical-rag-reranker.metadata-graph-index":
        raise ValueError("Unsupported graph index format")
    return payload


def relation_weight(entity: str, weights: dict[str, float]) -> float:
    relation = entity.split(":", 1)[0]
    return float(weights.get(relation, 0.05))


def graph_expand_run(
    *,
    base_run_path: Path,
    graph_index_path: Path,
    out_path: Path,
    run_name: str,
    top_k: int = 50,
    seed_top_n: int = 20,
    graph_weight: float = 0.65,
    rrf_k: int = 60,
    max_neighbors_per_entity: int = 80,
) -> Path:
    base_runs = read_trec_run(base_run_path)
    graph = load_graph_index(graph_index_path)
    doc_to_entities = graph["doc_to_entities"]
    entity_to_doc_ids = graph["entity_to_doc_ids"]
    weights = graph.get("relation_weights", {})

    expanded_runs: dict[str, list[tuple[str, float]]] = {}
    started = time.perf_counter()
    latencies_ms = []

    for qid, docs in base_runs.items():
        q_started = time.perf_counter()
        scores: dict[str, float] = {}
        for rank, (doc_id, _base_score) in enumerate(docs[:seed_top_n], start=1):
            base_rrf = 1.0 / (float(rrf_k) + float(rank))
            scores[doc_id] = scores.get(doc_id, 0.0) + base_rrf
            for entity in doc_to_entities.get(doc_id, []):
                boost = graph_weight * relation_weight(entity, weights) * base_rrf
                for neighbor_doc_id in entity_to_doc_ids.get(entity, [])[
                    :max_neighbors_per_entity
                ]:
                    scores[neighbor_doc_id] = scores.get(neighbor_doc_id, 0.0) + boost

        ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_k]
        expanded_runs[qid] = [(doc_id, float(score)) for doc_id, score in ranked]
        latencies_ms.append((time.perf_counter() - q_started) * 1000.0)

    write_trec_run(out_path, expanded_runs, run_name)
    latency_path = out_path.with_suffix(out_path.suffix + ".latency.json")
    latency_path.write_text(
        json.dumps(
            {
                "run_path": str(out_path),
                "base_run_path": str(base_run_path),
                "graph_index_path": str(graph_index_path),
                "num_queries": len(expanded_runs),
                "latencies_ms": latencies_ms,
                "total_latency_ms": (time.perf_counter() - started) * 1000.0,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return out_path

In [ ]:
def eval_existing_run(method: str, run_file: Path, index_path: Path) -> dict:
    overrides = base_overrides()
    overrides += [
        f"retrieval_run.index={index_path}",
        f"run.eval_retrieval.eval_queries={PROJECT_PROCESSED_DIR / 'eval_queries.jsonl'}",
        f"run.eval_retrieval.qrels={PROJECT_PROCESSED_DIR / 'qrels.tsv'}",
        f"run.eval_retrieval.out_run={run_file}",
        f"run.eval_retrieval.run_path={run_file}",
        f'run.eval_retrieval.ks="{KS}"',
        f"run.eval_retrieval.run_name={method}",
        f"run.eval_retrieval.retriever={method}",
    ]
    run_project("eval_retrieval", overrides)
    metrics = load_metrics(run_file)
    metrics.update(
        {
            "method": method,
            "retrieval_config": "graph_expansion",
            "index_path": str(index_path),
            "run_file": str(run_file),
            "stage": "graph_retrieval",
        }
    )
    return metrics


graph_index_path = Path(indices["graph_metadata"]["path"])
graph_rows = []
graph_bases = ["bm25", "hybrid_minilm"]
if "hybrid_trained_medcpt" in indices:
    graph_bases.append("hybrid_trained_medcpt")

for base_method in graph_bases:
    base_run = RETRIEVAL_RUN_DIR / f"{base_method}.trec"
    if not base_run.exists():
        print("Skipping graph expansion; missing base run:", base_run)
        continue
    method = f"graph_{base_method}"
    graph_run = RETRIEVAL_RUN_DIR / f"{method}.trec"
    graph_expand_run(
        base_run_path=base_run,
        graph_index_path=graph_index_path,
        out_path=graph_run,
        run_name=method,
        top_k=TOP_K,
    )
    graph_rows.append(eval_existing_run(method, graph_run, graph_index_path))

graph_df = pd.DataFrame(graph_rows)
if not graph_df.empty:
    display(
        graph_df[[col for col in display_cols if col in graph_df.columns]].sort_values(
            "NDCG@10", ascending=False
        )
    )

## Optional Cross-Encoder reranking

Укажите путь к checkpoint `*.ckpt`. Если checkpoint не задан, notebook честно пропускает reranker-секцию, но retrieval/graph результаты остаются пригодными для анализа.

In [ ]:
RERANKER_CHECKPOINT_PATH = os.environ.get("RERANKER_CHECKPOINT_PATH", "")
RERANK_TOP_N = 20
RERANKER_BATCH_SIZE = 16
RERANK_BASE_METHODS = ["bm25", "hybrid_minilm"]
if "hybrid_trained_medcpt" in indices:
    RERANK_BASE_METHODS.append("hybrid_trained_medcpt")
if (RETRIEVAL_RUN_DIR / "graph_hybrid_trained_medcpt.trec").exists():
    RERANK_BASE_METHODS.append("graph_hybrid_trained_medcpt")

print("Reranker checkpoint:", RERANKER_CHECKPOINT_PATH or "<not set>")
print("Rerank base methods:", RERANK_BASE_METHODS)

In [ ]:
def eval_reranked_run(base_method: str, checkpoint_path: Path) -> dict:
    base_run = RETRIEVAL_RUN_DIR / f"{base_method}.trec"
    if not base_run.exists():
        raise FileNotFoundError(base_run)

    method = f"{base_method}__reranked"
    out_run = RERANK_RUN_DIR / f"{method}.trec"
    comparison_report = RERANK_RUN_DIR / f"{method}.md"
    comparison_jsonl = RERANK_RUN_DIR / f"{method}.jsonl"

    overrides = base_overrides()
    overrides += [
        "retrieval=bm25",
        f"run.eval_reranked_retrieval.run_path={base_run}",
        f"run.eval_reranked_retrieval.eval_queries={PROJECT_PROCESSED_DIR / 'eval_queries.jsonl'}",
        f"run.eval_reranked_retrieval.qrels={PROJECT_PROCESSED_DIR / 'qrels.tsv'}",
        f"run.eval_reranked_retrieval.corpus_path={PROJECT_PROCESSED_DIR / 'corpus.jsonl'}",
        f"run.eval_reranked_retrieval.out_run={out_run}",
        f"run.eval_reranked_retrieval.reranker_checkpoint_path={checkpoint_path}",
        f"run.eval_reranked_retrieval.rerank_top_n={RERANK_TOP_N}",
        f"run.eval_reranked_retrieval.reranker_batch_size={RERANKER_BATCH_SIZE}",
        f'run.eval_reranked_retrieval.ks="{KS}"',
        f"run.eval_reranked_retrieval.run_name={method}",
        f"run.eval_reranked_retrieval.retriever={base_method}",
        f"run.eval_reranked_retrieval.comparison_report_path={comparison_report}",
        f"run.eval_reranked_retrieval.comparison_jsonl_path={comparison_jsonl}",
    ]
    run_project("eval_reranked_retrieval", overrides)

    comparison_path = out_run.with_suffix(out_run.suffix + ".comparison.json")
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    row = {f"rerank_{key}": value for key, value in comparison.items()}
    row.update(
        {
            "method": method,
            "base_method": base_method,
            "run_file": str(out_run),
            "stage": "reranked_retrieval",
            "reranker_checkpoint": str(checkpoint_path),
        }
    )
    return row


rerank_rows = []
if RERANKER_CHECKPOINT_PATH:
    checkpoint = Path(RERANKER_CHECKPOINT_PATH)
    if checkpoint.exists():
        for base_method in RERANK_BASE_METHODS:
            print("\n== Rerank", base_method, "==")
            rerank_rows.append(eval_reranked_run(base_method, checkpoint))
    else:
        print("Checkpoint path does not exist; skipping reranking:", checkpoint)
else:
    print("No reranker checkpoint configured; skipping reranking.")

rerank_df = pd.DataFrame(rerank_rows)
if not rerank_df.empty:
    display(rerank_df)

In [ ]:
all_rows = []
if not retrieval_df.empty:
    all_rows.extend(retrieval_df.to_dict("records"))
if not graph_df.empty:
    all_rows.extend(graph_df.to_dict("records"))
if not rerank_df.empty:
    all_rows.extend(rerank_df.to_dict("records"))

summary_df = pd.DataFrame(all_rows)
summary_csv = SUMMARY_DIR / "retrieval_benchmark_summary.csv"
summary_json = SUMMARY_DIR / "retrieval_benchmark_summary.json"
summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient="records", force_ascii=False, indent=2)

print("Saved summary CSV:", summary_csv)
print("Saved summary JSON:", summary_json)
if "NDCG@10" in summary_df.columns:
    cols = [col for col in display_cols if col in summary_df.columns]
    display(summary_df[cols].sort_values("NDCG@10", ascending=False))
else:
    display(summary_df)